# Chapter 1 · Agent Loop and Tool API Design

The same loop as the textbook, wired to the real Anthropic API. Every step is printed.

**Before you run:** add your key as a Colab secret named `ANTHROPIC_API_KEY` (key icon in the left sidebar) and enable notebook access.

In [ ]:
%pip install -q anthropic

In [ ]:
import os
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

import anthropic
client = anthropic.Anthropic()
MODEL = "claude-sonnet-5"  # fast and cheap; swap for "claude-opus-5" to compare

## Tools

The functions are tiny and read-only. The schemas are the contract the model sees. Note `strict: True` and `additionalProperties: False`.

In [ ]:
FINANCIALS = {
    ("DE", "Q2-2026"):   {"name": "Deere", "revenue": 13.8e9, "yoy": 0.064},
    ("CAT", "Q2-2026"):  {"name": "Caterpillar", "revenue": 16.9e9, "yoy": 0.031},
    ("NVDA", "Q2-2026"): {"name": "NVIDIA", "revenue": 52.4e9, "yoy": 0.58},
}
PRICES = {"DE": 512.40, "CAT": 398.15, "NVDA": 181.22}

def get_financials(ticker, period="Q2-2026"):
    row = FINANCIALS.get((ticker, period))
    return {**row, "ticker": ticker, "period": period} if row else {"error": f"no data for {ticker} {period}"}

TOOLS = {"get_financials": get_financials}

TOOL_SCHEMAS = [
    {"name": "get_financials",
     "description": "Quarterly revenue and YoY growth for one ticker. Use only when the user asks about revenue, growth, or earnings.",
     "strict": True,
     "input_schema": {"type": "object",
                      "properties": {"ticker": {"type": "string", "enum": ["DE", "CAT", "NVDA"]},
                                     "period": {"type": "string", "enum": ["Q2-2026"]}},
                      "required": ["ticker", "period"], "additionalProperties": False}},
]

## The loop

Identical in shape to the textbook. The only differences are the message format the API expects (`tool_use` and `tool_result` content blocks) and that a reply can contain several tool calls at once, which we run and return together.

In [ ]:
import json, time

def agent(question, tools=TOOLS, schemas=TOOL_SCHEMAS, max_steps=6):
    messages = [{"role": "user", "content": question}]
    log = []
    for step in range(max_steps):
        response = client.messages.create(
            model=MODEL, max_tokens=4096, tools=schemas, messages=messages,
        )
        tool_uses = [b for b in response.content if b.type == "tool_use"]
        if response.stop_reason != "tool_use" or not tool_uses:
            text = next((b.text for b in response.content if b.type == "text"), "")
            log.append({"step": step, "kind": "text", "text": text})
            print(f"step {step}: text -> {text}")
            return text, log

        messages.append({"role": "assistant", "content": response.content})
        results = []
        for tu in tool_uses:
            t0 = time.time()
            try:
                result = tools[tu.name](**tu.input)
            except Exception as e:
                result = {"error": f"{type(e).__name__}: {e}"}
            ms = round((time.time() - t0) * 1000, 2)
            log.append({"step": step, "kind": "tool_call", "tool": tu.name, "args": tu.input, "result": result, "ms": ms})
            print(f"step {step}: tool_call {tu.name}({json.dumps(tu.input)}) -> {json.dumps(result)[:100]}")
            results.append({"type": "tool_result", "tool_use_id": tu.id, "content": json.dumps(result)})
        messages.append({"role": "user", "content": results})

    log.append({"step": max_steps, "kind": "budget_exhausted"})
    return "Stopped: step budget exhausted.", log

answer, log = agent("What was Deere's revenue growth last quarter?")

## A live business example: refund triage

The same loop, now doing a job a business pays for. Two read-only tools: `get_order` and `search_docs`. Deliberately no `issue_refund` tool: the model recommends, a human approves.

In [ ]:
ORDERS = {
    "4471": {"customer": "Priya Natarajan", "item": "Standing desk", "amount": 449.00, "days_since_purchase": 9,  "used": False, "receipt": True},
    "4488": {"customer": "Marcus Bell",     "item": "Office chair",  "amount": 289.00, "days_since_purchase": 21, "used": False, "receipt": True},
    "4502": {"customer": "Elena Ruiz",      "item": "Monitor arm",   "amount": 79.00,  "days_since_purchase": 4,  "used": False, "receipt": False},
    "4519": {"customer": "Tom Okafor",      "item": "Desk lamp",     "amount": 59.00,  "days_since_purchase": 6,  "used": True,  "receipt": True},
}
DOCS = [
    {"id": "refund-policy-1", "text": "Refunds are issued within 14 days of purchase for unused items; a receipt is required."},
    {"id": "refund-policy-2", "text": "Refunds after 14 days are issued as store credit only, at the manager's discretion."},
    {"id": "shipping-1", "text": "Standard shipping takes 3-5 business days; expedited shipping arrives in 1-2 business days."},
    {"id": "warranty-1", "text": "All equipment carries a one-year limited warranty covering manufacturing defects."},
]

def get_order(order_id):
    row = ORDERS.get(order_id.lstrip("#"))
    return {**row, "order_id": order_id} if row else {"error": f"no order {order_id}"}

def search_docs(query, k=3):
    words = set(query.lower().split())
    scored = sorted(DOCS, key=lambda d: -sum(w in d["text"].lower() for w in words))
    return scored[:k]

REFUND_TOOLS = {"get_order": get_order, "search_docs": search_docs}
REFUND_SCHEMAS = [
    {"name": "get_order", "strict": True,
     "description": "Look up one customer order by id. Use when a question names an order number.",
     "input_schema": {"type": "object", "properties": {"order_id": {"type": "string", "pattern": "^[0-9]{4}$"}},
                      "required": ["order_id"], "additionalProperties": False}},
    {"name": "search_docs", "strict": True,
     "description": "Keyword search over company policy documents. Do not call with an empty or one-word query.",
     "input_schema": {"type": "object", "properties": {"query": {"type": "string", "minLength": 4}, "k": {"type": "integer", "minimum": 1, "maximum": 5}},
                      "required": ["query", "k"], "additionalProperties": False}},
]

SYSTEM = ("You are a support-desk assistant for an office-furniture retailer. Look up the order, then the policy, "
          "then give ONE recommendation in the form 'RECOMMEND: <APPROVE|STORE CREDIT|HOLD|DECLINE>. <reason>' "
          "and cite the policy chunk id in brackets like [source: refund-policy-1]. You cannot issue refunds; a human approves.")

In [ ]:
def agent_with_system(question, tools, schemas, system, max_steps=6):
    messages = [{"role": "user", "content": question}]
    log = []
    for step in range(max_steps):
        response = client.messages.create(model=MODEL, max_tokens=4096, system=system, tools=schemas, messages=messages)
        tool_uses = [b for b in response.content if b.type == "tool_use"]
        if response.stop_reason != "tool_use" or not tool_uses:
            text = next((b.text for b in response.content if b.type == "text"), "")
            log.append({"step": step, "kind": "text", "text": text})
            return text, log
        messages.append({"role": "assistant", "content": response.content})
        results = []
        for tu in tool_uses:
            try:
                result = tools[tu.name](**tu.input)
            except Exception as e:
                result = {"error": f"{type(e).__name__}: {e}"}
            log.append({"step": step, "kind": "tool_call", "tool": tu.name, "args": tu.input, "result": result})
            print(f"step {step}: {tu.name}({json.dumps(tu.input)})")
            results.append({"type": "tool_result", "tool_use_id": tu.id, "content": json.dumps(result)})
        messages.append({"role": "user", "content": results})
    return "Stopped: step budget exhausted.", log

for oid in ["4471", "4488", "4502", "4519"]:
    answer, log = agent_with_system(f"Customer is asking for a refund on order #{oid}. What should we do?", REFUND_TOOLS, REFUND_SCHEMAS, SYSTEM)
    print(f"#{oid} -> {answer}\n")

Compare with the textbook's mock run. Did the real model look up the order before the policy? Did every recommendation carry a `[source: …]` tag? If not, the fix belongs in the tool descriptions or the system prompt, and you should be able to say which.

## Exercise

1. Run the refund-triage scenario above for all four orders and compare the tool sequence and recommendations with the textbook's mock run.
2. Add `get_price(ticker)` to `TOOLS` and a closed schema for it to `TOOL_SCHEMAS`, then run the question below. Expect four tool calls and one text answer.
3. Add the total elapsed time of the whole run to the final log entry.
4. Write three sentences: one tool call the model made that you would not have made, and the schema change that would prevent it.

In [ ]:
# 1. your get_price tool and schema here


# 3.
answer, log = agent("Is Deere's revenue growth better than Caterpillar's, and what are both trading at?")
print()
print(json.dumps(log, indent=1, default=str))

## Optional: compare models

Set `MODEL = "claude-opus-5"` and rerun the exercise question. Count the tool calls and read the final answer. Which one would you ship, and why?